# MiFID Migration Audit Tables

This notebook creates audit and validation tables for the MiFID staging migration. These tables track:
- **Execution state** — row counts and timestamps for each staging table per run
- **Data quality** — key integrity, duplicate checks, referential consistency
- **Cross-module integrity** — join coverage between dependent staging objects

**Audit tables created:**
- `bi_output_regtechops_mifid_staging_audit_log` — Per-run row counts and pass/fail status
- `bi_output_regtechops_mifid_quality_audit` — Data quality check results
- `bi_output_regtechops_mifid_xref_audit` — Cross-module referential integrity

**Prerequisites:** All staging notebooks (01–05) must have been run first.

In [0]:
dbutils.widgets.text("report_date", "2026-06-09", "Report Date (YYYY-MM-DD)")
report_date = dbutils.widgets.get("report_date")
print(f"Running MiFID Audit Tables for report_date = {report_date}")

In [0]:
%sql
-- Staging execution audit: captures row counts and timestamps for each staging table per run

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_mifid_staging_audit_log
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/mifid_staging_audit_log'
AS
SELECT
  CAST('${report_date}' AS DATE) AS report_date,
  current_timestamp() AS audit_timestamp,
  table_name,
  row_count,
  CASE WHEN row_count > 0 THEN 'PASS' ELSE 'EMPTY' END AS status
FROM (
  SELECT 'reg_currencyprice_ext' AS table_name, (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_currencyprice_ext) AS row_count
  UNION ALL SELECT 'reg_ext_dailymaxprices', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_dailymaxprices)
  UNION ALL SELECT 'reg_ext_currencypricemaxdatewithsplit', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_currencypricemaxdatewithsplit)
  UNION ALL SELECT 'reg_ext_historysplitratio', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_historysplitratio)
  UNION ALL SELECT 'reg_ext_trade_getinstrument', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_trade_getinstrument)
  UNION ALL SELECT 'reg_ext_trade_instrumentmetadata', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_trade_instrumentmetadata)
  UNION ALL SELECT 'reg_ext_dictionarycurrency', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_dictionarycurrency)
  UNION ALL SELECT 'reg_ext_dictionarycurrencytype', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_dictionarycurrencytype)
  UNION ALL SELECT 'reg_ext_hedgeexecutionlog', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_hedgeexecutionlog)
  UNION ALL SELECT 'reg_ext_hedgehbcexecutionlog', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_hedgehbcexecutionlog)
  UNION ALL SELECT 'reg_ext_hedgehbcorderlog', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_hedgehbcorderlog)
  UNION ALL SELECT 'reg_instruments_ext', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_instruments_ext)
  UNION ALL SELECT 'reg_hedgeservertoliquidityaccount_ext', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_hedgeservertoliquidityaccount_ext)
  UNION ALL SELECT 'reg_liquidtyacount_ext', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_liquidtyacount_ext)
  UNION ALL SELECT 'reg_ext_liquidityaccountid', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_liquidityaccountid)
  UNION ALL SELECT 'reg_ext_liquidityproviders', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_liquidityproviders)
  UNION ALL SELECT 'reg_migrationinout_population', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_migrationinout_population)
  UNION ALL SELECT 'reg_regulationinoutdailydata', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_regulationinoutdailydata)
  UNION ALL SELECT 'mifid2_ext_positionchangelog', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_positionchangelog)
  UNION ALL SELECT 'mifid2_ext_mirror', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_mirror)
  UNION ALL SELECT 'mifid2_ext_hedgeexecutionlog', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_hedgeexecutionlog)
  UNION ALL SELECT 'mifid2_ext_customer', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_customer)
  UNION ALL SELECT 'mifid2_ext_regchange_customer', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_regchange_customer)
  UNION ALL SELECT 'mifid2_ext_position', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_position)
  UNION ALL SELECT 'mifid2_ext_regchange_position', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_regchange_position)
  UNION ALL SELECT 'mifid2_report_trade_population', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_report_trade_population)
  UNION ALL SELECT 'mifid2_report', (SELECT COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_report)
)

In [0]:
%sql
-- Data quality audit: key integrity and referential checks

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_mifid_quality_audit
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/mifid_quality_audit'
AS
SELECT
  CAST('${report_date}' AS DATE) AS report_date,
  current_timestamp() AS audit_timestamp,
  check_name,
  check_result,
  CASE WHEN check_result = 0 THEN 'PASS' ELSE 'FAIL' END AS status,
  detail
FROM (
  -- Check 1: Duplicate LiquidityAccountIDs in ext
  SELECT 'dup_liquidityaccount_ext' AS check_name,
    (SELECT COUNT(*) - COUNT(DISTINCT LiquidityAccountID) FROM main.regtech_ops_stg.bi_output_regtechops_reg_liquidtyacount_ext) AS check_result,
    'Duplicate LiquidityAccountID in reg_liquidtyacount_ext' AS detail
  UNION ALL
  -- Check 2: Instruments in hedge log not in instruments_ext
  SELECT 'orphan_instrument_in_hedge',
    (SELECT COUNT(DISTINCT h.InstrumentID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_hedgeexecutionlog h
     WHERE h.InstrumentID NOT IN (SELECT InstrumentID FROM main.regtech_ops_stg.bi_output_regtechops_reg_instruments_ext)),
    'Distinct InstrumentIDs in hedge execution log missing from instruments_ext'
  UNION ALL
  -- Check 3: Duplicate InstrumentIDs in trade_getinstrument
  SELECT 'dup_instrumentid_getinstrument',
    (SELECT COUNT(*) - COUNT(DISTINCT InstrumentID) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_trade_getinstrument),
    'Duplicate InstrumentID in reg_ext_trade_getinstrument'
  UNION ALL
  -- Check 4: Customer deduplication check (should be 1 row per CID)
  SELECT 'customer_dedup_ratio',
    (SELECT COUNT(*) - COUNT(DISTINCT CID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_customer),
    'Excess rows beyond 1-per-CID in mifid2_ext_customer (0 = deduplicated)'
  UNION ALL
  -- Check 5: Trade population - positions without instrument match
  SELECT 'trade_pop_orphan_instruments',
    (SELECT COUNT(DISTINCT t.InstrumentID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_report_trade_population t
     WHERE t.InstrumentID NOT IN (SELECT InstrumentID FROM main.regtech_ops_stg.bi_output_regtechops_reg_instruments_ext)),
    'Distinct InstrumentIDs in trade_population missing from instruments_ext'
)

In [0]:
%sql
-- Cross-module referential integrity audit
CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_mifid_xref_audit
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/mifid_xref_audit'
AS
SELECT
  CAST('${report_date}' AS DATE) AS report_date,
  current_timestamp() AS audit_timestamp,
  relationship,
  left_table,
  right_table,
  total_left_keys,
  matched_keys,
  unmatched_keys,
  ROUND(matched_keys * 100.0 / NULLIF(total_left_keys, 0), 2) AS match_pct,
  CASE WHEN unmatched_keys = 0 THEN 'FULL_MATCH' 
       WHEN matched_keys * 100.0 / NULLIF(total_left_keys, 0) >= 95 THEN 'ACCEPTABLE'
       ELSE 'REVIEW' END AS status
FROM (
  -- Hedge execution: LiquidityAccountID -> LiquidityAccount_Ext
  SELECT 'hedge_to_liquidity' AS relationship,
    'mifid2_ext_hedgeexecutionlog' AS left_table,
    'reg_liquidtyacount_ext' AS right_table,
    (SELECT COUNT(DISTINCT LiquidityAccountID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_hedgeexecutionlog) AS total_left_keys,
    (SELECT COUNT(DISTINCT h.LiquidityAccountID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_hedgeexecutionlog h
     WHERE h.LiquidityAccountID IN (SELECT LiquidityAccountID FROM main.regtech_ops_stg.bi_output_regtechops_reg_liquidtyacount_ext)) AS matched_keys,
    (SELECT COUNT(DISTINCT h.LiquidityAccountID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_hedgeexecutionlog h
     WHERE h.LiquidityAccountID NOT IN (SELECT LiquidityAccountID FROM main.regtech_ops_stg.bi_output_regtechops_reg_liquidtyacount_ext)) AS unmatched_keys
  UNION ALL
  -- Hedge execution: InstrumentID -> Trade_GetInstrument
  SELECT 'hedge_to_instruments',
    'mifid2_ext_hedgeexecutionlog',
    'reg_ext_trade_getinstrument',
    (SELECT COUNT(DISTINCT InstrumentID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_hedgeexecutionlog),
    (SELECT COUNT(DISTINCT h.InstrumentID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_hedgeexecutionlog h
     WHERE h.InstrumentID IN (SELECT InstrumentID FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_trade_getinstrument)),
    (SELECT COUNT(DISTINCT h.InstrumentID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_hedgeexecutionlog h
     WHERE h.InstrumentID NOT IN (SELECT InstrumentID FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_trade_getinstrument))
  UNION ALL
  -- LiquidityProviders: LiquidityProviderID referential check
  SELECT 'liquidity_ext_to_providers',
    'reg_liquidtyacount_ext',
    'reg_ext_liquidityproviders',
    (SELECT COUNT(DISTINCT LiquidityProviderID) FROM main.regtech_ops_stg.bi_output_regtechops_reg_liquidtyacount_ext),
    (SELECT COUNT(DISTINCT la.LiquidityProviderID) FROM main.regtech_ops_stg.bi_output_regtechops_reg_liquidtyacount_ext la
     WHERE la.LiquidityProviderID IN (SELECT LiquidityProviderID FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_liquidityproviders)),
    (SELECT COUNT(DISTINCT la.LiquidityProviderID) FROM main.regtech_ops_stg.bi_output_regtechops_reg_liquidtyacount_ext la
     WHERE la.LiquidityProviderID NOT IN (SELECT LiquidityProviderID FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_liquidityproviders))
  UNION ALL
  -- Mirror: ParentCID in BackOfficeCustomer
  SELECT 'mirror_to_backoffice',
    'mifid2_ext_mirror',
    'history_backofficecustomer',
    (SELECT COUNT(DISTINCT ParentCID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_mirror),
    (SELECT COUNT(DISTINCT m.ParentCID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_mirror m
     WHERE m.ParentCID IN (SELECT DISTINCT CID FROM main.general.bronze_etoro_history_backofficecustomer)),
    (SELECT COUNT(DISTINCT m.ParentCID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_mirror m
     WHERE m.ParentCID NOT IN (SELECT DISTINCT CID FROM main.general.bronze_etoro_history_backofficecustomer))
  UNION ALL
  -- Position CIDs all in Customer staging
  SELECT 'position_to_customer',
    'mifid2_ext_position',
    'mifid2_ext_customer',
    (SELECT COUNT(DISTINCT CID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_position),
    (SELECT COUNT(DISTINCT p.CID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_position p
     WHERE p.CID IN (SELECT CID FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_customer)),
    (SELECT COUNT(DISTINCT p.CID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_position p
     WHERE p.CID NOT IN (SELECT CID FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_customer))
  UNION ALL
  -- Position InstrumentID in instruments_ext
  SELECT 'position_to_instruments',
    'mifid2_ext_position',
    'reg_instruments_ext',
    (SELECT COUNT(DISTINCT InstrumentID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_position),
    (SELECT COUNT(DISTINCT p.InstrumentID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_position p
     WHERE p.InstrumentID IN (SELECT InstrumentID FROM main.regtech_ops_stg.bi_output_regtechops_reg_instruments_ext)),
    (SELECT COUNT(DISTINCT p.InstrumentID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_position p
     WHERE p.InstrumentID NOT IN (SELECT InstrumentID FROM main.regtech_ops_stg.bi_output_regtechops_reg_instruments_ext))
)

In [0]:
%sql
-- Summary view: all audit results
SELECT '--- STAGING AUDIT LOG ---' AS section, '' AS detail, '' AS status, 0 AS value
UNION ALL
SELECT table_name, CAST(row_count AS STRING), status, row_count FROM main.regtech_ops_stg.bi_output_regtechops_mifid_staging_audit_log
UNION ALL
SELECT '--- QUALITY CHECKS ---', '', '', 0
UNION ALL
SELECT check_name, detail, status, check_result FROM main.regtech_ops_stg.bi_output_regtechops_mifid_quality_audit
UNION ALL
SELECT '--- XREF INTEGRITY ---', '', '', 0
UNION ALL
SELECT relationship, CONCAT(CAST(match_pct AS STRING), '% match (', CAST(matched_keys AS STRING), '/', CAST(total_left_keys AS STRING), ')'), status, unmatched_keys
FROM main.regtech_ops_stg.bi_output_regtechops_mifid_xref_audit
ORDER BY section